In [49]:
import numpy as np
import pandas as pd
import pickle
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

from sklearn.model_selection import cross_val_score
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.metrics import  accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,average_precision_score

In [2]:
x_train = np.load(r"../data/processed/x_train_fea.npy")
x_test = np.load(r"../data/processed/x_test_fea.npy")
y_train = np.load(r"../data/processed/y_train_fea.npy")
y_test = np.load(r"../data/processed/y_test_fea.npy")

In [3]:
print("X_train:", x_train.shape)
print("X_test:", x_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5634, 48)
X_test: (1409, 48)
y_train: (5634,)
y_test: (1409,)


In [4]:
performance ={
    "model":[],
    "acc":[],
    "precision":[],
    "recall":[],
    "f1":[],
    "roc":[],
    "pr_auc":[]
}
def add(model,acc,precision,recall,f1,roc_auc,pr_auc):
    performance["model"].append(model)
    performance["acc"].append(f"{acc:.2f}")
    performance["precision"].append(f"{precision:.2f}")
    performance["recall"].append(f"{recall:.2f}")
    performance["f1"].append(f"{f1:.2f}")
    performance["roc"].append(f"{roc_auc:.2f}")
    performance["pr_auc"].append(f"{pr_auc:.2f}")
    return performance

In [5]:
def metrics(y_pred,y_prob):
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall =  recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    pr_auc =  average_precision_score(y_test, y_prob)
    return accuracy,precision,recall,f1,roc,pr_auc

In [6]:
#baseline model

In [7]:
LRM = LogisticRegression(max_iter=1000)
LRM.fit(x_train,y_train)
y_pred = LRM.predict(x_test)

In [8]:
y_prob = LRM.predict_proba(x_test)[:, 1]
y_prob

array([0.04721206, 0.72277098, 0.04911572, ..., 0.16101216, 0.00743453,
       0.00702506], shape=(1409,))

In [9]:
accuracy,precision,recall,f1,roc,pr_auc = metrics(y_pred,y_prob)
print(add("Logistic Regression",accuracy,precision,recall,f1,roc,pr_auc))

{'model': ['Logistic Regression'], 'acc': ['0.80'], 'precision': ['0.66'], 'recall': ['0.52'], 'f1': ['0.58'], 'roc': ['0.84'], 'pr_auc': ['0.64']}


In [10]:
#decison tree classifier

In [11]:
DT_MODEL = DecisionTreeClassifier(random_state=42)
DT_MODEL.fit(x_train, y_train)

y_pred = DT_MODEL.predict(x_test)
y_prob = DT_MODEL.predict_proba(x_test)[:, 1]

In [12]:
y_pred = DT_MODEL.predict(x_test)
y_prob = DT_MODEL.predict_proba(x_test)[:, 1]

In [13]:
accuracy,precision,recall,f1,roc,pr_auc = metrics(y_pred,y_prob)
print(add("Decision Tree",accuracy,precision,recall,f1,roc,pr_auc))

{'model': ['Logistic Regression', 'Decision Tree'], 'acc': ['0.80', '0.72'], 'precision': ['0.66', '0.48'], 'recall': ['0.52', '0.52'], 'f1': ['0.58', '0.50'], 'roc': ['0.84', '0.66'], 'pr_auc': ['0.64', '0.38']}


In [14]:
#KNN

In [15]:
KNN = KNeighborsClassifier(n_neighbors=5)

KNN.fit(x_train, y_train)

y_pred = KNN.predict(x_test)
y_prob = KNN.predict_proba(x_test)[:, 1]

In [16]:
y_pred = KNN.predict(x_test)
y_prob = KNN.predict_proba(x_test)[:, 1]

In [17]:
accuracy,precision,recall,f1,roc,pr_auc = metrics(y_pred,y_prob)
print(add("KNN",accuracy,precision,recall,f1,roc,pr_auc))

{'model': ['Logistic Regression', 'Decision Tree', 'KNN'], 'acc': ['0.80', '0.72', '0.78'], 'precision': ['0.66', '0.48', '0.59'], 'recall': ['0.52', '0.52', '0.53'], 'f1': ['0.58', '0.50', '0.56'], 'roc': ['0.84', '0.66', '0.78'], 'pr_auc': ['0.64', '0.38', '0.52']}


In [18]:
#random forest

In [19]:
random_forest = RandomForestClassifier(n_estimators=100,random_state=42)

random_forest.fit(x_train, y_train)

rf_pred = random_forest.predict(x_test)
rf_prob = random_forest.predict_proba(x_test)[:, 1]

In [20]:
y_pred = random_forest.predict(x_test)
y_prob = random_forest.predict_proba(x_test)[:, 1]

In [21]:
accuracy,precision,recall,f1,roc,pr_auc = metrics(y_pred,y_prob)
print(add("Random forest",accuracy,precision,recall,f1,roc,pr_auc))

{'model': ['Logistic Regression', 'Decision Tree', 'KNN', 'Random forest'], 'acc': ['0.80', '0.72', '0.78', '0.78'], 'precision': ['0.66', '0.48', '0.59', '0.61'], 'recall': ['0.52', '0.52', '0.53', '0.51'], 'f1': ['0.58', '0.50', '0.56', '0.56'], 'roc': ['0.84', '0.66', '0.78', '0.82'], 'pr_auc': ['0.64', '0.38', '0.52', '0.61']}


In [22]:
#gradientboosting classifer

In [23]:
gb_model = GradientBoostingClassifier(random_state=42)

gb_model.fit(x_train, y_train)

y_pred = gb_model.predict(x_test)
y_prob = gb_model.predict_proba(x_test)[:, 1]

In [24]:
y_pred = gb_model.predict(x_test)
y_prob = gb_model.predict_proba(x_test)[:, 1]

In [25]:
accuracy,precision,recall,f1,roc,pr_auc = metrics(y_pred,y_prob)
print(add("gradient boosting",accuracy,precision,recall,f1,roc,pr_auc))

{'model': ['Logistic Regression', 'Decision Tree', 'KNN', 'Random forest', 'gradient boosting'], 'acc': ['0.80', '0.72', '0.78', '0.78', '0.80'], 'precision': ['0.66', '0.48', '0.59', '0.61', '0.67'], 'recall': ['0.52', '0.52', '0.53', '0.51', '0.52'], 'f1': ['0.58', '0.50', '0.56', '0.56', '0.59'], 'roc': ['0.84', '0.66', '0.78', '0.82', '0.85'], 'pr_auc': ['0.64', '0.38', '0.52', '0.61', '0.66']}


xgboost

In [26]:
xgb_model = XGBClassifier(random_state=42,eval_metric='logloss')

xgb_model.fit(x_train, y_train)

y_pred = xgb_model.predict(x_test)
y_prob = xgb_model.predict_proba(x_test)[:, 1]

In [27]:
y_pred = xgb_model.predict(x_test)
y_prob = xgb_model.predict_proba(x_test)[:, 1]

In [28]:
accuracy,precision,recall,f1,roc,pr_auc = metrics(y_pred,y_prob)
print(add("Xg Boost",accuracy,precision,recall,f1,roc,pr_auc))

{'model': ['Logistic Regression', 'Decision Tree', 'KNN', 'Random forest', 'gradient boosting', 'Xg Boost'], 'acc': ['0.80', '0.72', '0.78', '0.78', '0.80', '0.77'], 'precision': ['0.66', '0.48', '0.59', '0.61', '0.67', '0.59'], 'recall': ['0.52', '0.52', '0.53', '0.51', '0.52', '0.49'], 'f1': ['0.58', '0.50', '0.56', '0.56', '0.59', '0.54'], 'roc': ['0.84', '0.66', '0.78', '0.82', '0.85', '0.82'], 'pr_auc': ['0.64', '0.38', '0.52', '0.61', '0.66', '0.61']}


In [29]:
performance

{'model': ['Logistic Regression',
  'Decision Tree',
  'KNN',
  'Random forest',
  'gradient boosting',
  'Xg Boost'],
 'acc': ['0.80', '0.72', '0.78', '0.78', '0.80', '0.77'],
 'precision': ['0.66', '0.48', '0.59', '0.61', '0.67', '0.59'],
 'recall': ['0.52', '0.52', '0.53', '0.51', '0.52', '0.49'],
 'f1': ['0.58', '0.50', '0.56', '0.56', '0.59', '0.54'],
 'roc': ['0.84', '0.66', '0.78', '0.82', '0.85', '0.82'],
 'pr_auc': ['0.64', '0.38', '0.52', '0.61', '0.66', '0.61']}

In [30]:
performance_df = pd.DataFrame(performance)

In [31]:
performance_df

,model,acc,precision,recall,f1,roc,pr_auc
0,Logistic Regression,0.80,0.66,0.52,0.58,0.84,0.64
1,Decision Tree,0.72,0.48,0.52,0.50,0.66,0.38
2,KNN,0.78,0.59,0.53,0.56,0.78,0.52
3,Random forest,0.78,0.61,0.51,0.56,0.82,0.61
4,gradient boosting,0.80,0.67,0.52,0.59,0.85,0.66
5,Xg Boost,0.77,0.59,0.49,0.54,0.82,0.61


In [32]:
#best model
#gradient boost model

model improvement

In [33]:
#cross validation
gb_model_improvemt = GradientBoostingClassifier(random_state=42)

cv_scores = cross_val_score(
    gb_model_improvemt,
    x_train,
    y_train,
    cv=5,
    scoring='f1'
)

print("F1 scores:", cv_scores)
print("Mean F1:", cv_scores.mean())

F1 scores: [0.62773723 0.5681382  0.5973025  0.55390335 0.5648855 ]
Mean F1: 0.5823933537559585


class weight balence

In [34]:
sample_weights = compute_sample_weight(class_weight='balanced',y=y_train)

In [35]:
gb_weighted = GradientBoostingClassifier(random_state=42)

gb_weighted.fit(x_train,y_train,sample_weight=sample_weights)

y_pred_weighted = gb_weighted.predict(x_test)
y_prob_weighted = gb_weighted.predict_proba(x_test)[:, 1]

In [36]:
print("Accuracy :", accuracy_score(y_test, y_pred_weighted))
print("Precision:", precision_score(y_test, y_pred_weighted))
print("Recall   :", recall_score(y_test, y_pred_weighted))
print("F1 Score :", f1_score(y_test, y_pred_weighted))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_weighted))
print("PR-AUC   :", average_precision_score(y_test, y_prob_weighted))

Accuracy : 0.7466288147622427
Precision: 0.5148861646234676
Recall   : 0.786096256684492
F1 Score : 0.6222222222222222
ROC-AUC  : 0.8416686042005735
PR-AUC   : 0.6594707495436567


threshold adjustment

In [ ]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

for threshold in thresholds:

    y_pred_threshold = (y_prob_weighted >= threshold).astype(int)

    precision = precision_score(y_test, y_pred_threshold)
    recall = recall_score(y_test, y_pred_threshold)
    f1 = f1_score(y_test, y_pred_threshold)

    print(
        f"Threshold: {threshold} | "
        f"Precision: {precision:.3f} | "
        f"Recall: {recall:.3f} | "
        f"F1: {f1:.3f}")

Threshold: 0.3 | Precision: 0.436 | Recall: 0.912 | F1: 0.590
Threshold: 0.4 | Precision: 0.476 | Recall: 0.845 | F1: 0.609
Threshold: 0.5 | Precision: 0.515 | Recall: 0.786 | F1: 0.622
Threshold: 0.6 | Precision: 0.565 | Recall: 0.690 | F1: 0.621
Threshold: 0.7 | Precision: 0.627 | Recall: 0.567 | F1: 0.596


random search

In [38]:
gb_model = GradientBoostingClassifier(random_state=42)

param_grid = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [2, 3, 4],
    'min_samples_split': [2, 5, 10]
}

random_search = RandomizedSearchCV(
    estimator=gb_model,
    param_distributions=param_grid,
    n_iter=10,
    scoring='f1',
    cv=5,
    random_state=42,
    n_jobs=-1
)

random_search.fit(x_train, y_train)

print("Best Parameters:")
print(random_search.best_params_)

print("\nBest CV F1:")
print(random_search.best_score_)

Best Parameters:
{'n_estimators': 100, 'min_samples_split': 10, 'max_depth': 3, 'learning_rate': 0.1}

Best CV F1:
0.5796426104979762


In [39]:
tuned_gb = random_search.best_estimator_

tuned_pred = tuned_gb.predict(x_test)
tuned_prob = tuned_gb.predict_proba(x_test)[:, 1]

In [40]:
print("Accuracy :", accuracy_score(y_test, tuned_pred))
print("Precision:", precision_score(y_test, tuned_pred))
print("Recall   :", recall_score(y_test, tuned_pred))
print("F1 Score :", f1_score(y_test, tuned_pred))
print("ROC-AUC  :", roc_auc_score(y_test, tuned_prob))
print("PR-AUC   :", average_precision_score(y_test, tuned_prob))

Accuracy : 0.7984386089425124
Precision: 0.652027027027027
Recall   : 0.516042780748663
F1 Score : 0.5761194029850746
ROC-AUC  : 0.8438916014363584
PR-AUC   : 0.6608125096006481


feature selection

In [41]:
with open("encoder.pkl", "rb") as file:
    encoder = pickle.load(file)
    
with open("scaler.pkl", "rb") as file:
    scaler = pickle.load(file)

In [42]:
numeric_feature_names = scaler.get_feature_names_out()

categorical_feature_names = encoder.get_feature_names_out()

feature_names = list(numeric_feature_names) + list(categorical_feature_names)

print("Total feature names:", len(feature_names))

Total feature names: 48


In [ ]:
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': gb_weighted.feature_importances_
})

feature_importance = feature_importance.sort_values('Importance',ascending=False)

feature_importance.head(15)

,Feature,Importance
10,contract_risk,0.468574
12,service_combination_risk,0.105019
1,tenure,0.100247
20,InternetService_Fiber optic,0.060685
2,MonthlyCharges,0.056800
3,TotalCharges,0.054471
9,charge_to_tenure_ratio,0.037541
11,payment_method_risk,0.020935
35,Contract_Two year,0.018032
36,PaperlessBilling_Yes,0.012607


In [44]:
top_15_features = feature_importance.head(15)['Feature'].tolist()

print(top_15_features)

['contract_risk', 'service_combination_risk', 'tenure', 'InternetService_Fiber optic', 'MonthlyCharges', 'TotalCharges', 'charge_to_tenure_ratio', 'payment_method_risk', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Electronic check', 'StreamingMovies_Yes', 'Contract_One year', 'Dependents_Yes', 'OnlineBackup_Yes']


In [45]:
top_15_indices = [feature_names.index(feature)for feature in top_15_features]

print(top_15_indices)

[10, 12, 1, 20, 2, 3, 9, 11, 35, 36, 38, 33, 34, 16, 25]


In [46]:
X_train_top15 = x_train[:, top_15_indices]
X_test_top15 = x_test[:, top_15_indices]

print("Original:", x_train.shape)
print("Selected:", X_train_top15.shape)

Original: (5634, 48)
Selected: (5634, 15)


In [47]:
sample_weights_top15 = compute_sample_weight(class_weight='balanced',y=y_train)

gb_top15 = GradientBoostingClassifier(random_state=42)

gb_top15.fit(X_train_top15, y_train, sample_weight=sample_weights_top15)

y_pred_top15 = gb_top15.predict(X_test_top15)
y_prob_top15 = gb_top15.predict_proba(X_test_top15)[:, 1]

In [48]:
print("Accuracy :", accuracy_score(y_test, y_pred_top15))
print("Precision:", precision_score(y_test, y_pred_top15))
print("Recall   :", recall_score(y_test, y_pred_top15))
print("F1 Score :", f1_score(y_test, y_pred_top15))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_top15))
print("PR-AUC   :", average_precision_score(y_test, y_prob_top15))

Accuracy : 0.7444996451383961
Precision: 0.5121107266435986
Recall   : 0.7914438502673797
F1 Score : 0.6218487394957983
ROC-AUC  : 0.8410821772714355
PR-AUC   : 0.6583417317075068


In [ ]:
#the feature reduction is completed the model perform same as the with the total features

In [52]:
joblib.dump(gb_top15, r"../models/model.pkl")

joblib.dump(top_15_features, r"../models/top_15_features.pkl")

joblib.dump(top_15_indices, r"../models/top_15_indices.pkl")

['../models/top_15_indices.pkl']